In [ ]:
import streamlit as st
import requests
import pandas as pd

st.set_page_config(page_title="Cricbuzz Dashboard", layout="wide")

# ---------------------------
# Fetch Live Data
# ---------------------------
@st.cache_data(ttl=60)
def fetch_matches():
    url = "https://www.cricbuzz.com/api/cricket-match/live"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        return data
    return {}

data = fetch_matches()

# ---------------------------
# Process Data
# ---------------------------
matches = []
teams = set()
venues = set()
team_wins = {}
venue_count = {}

if "typeMatches" in data:
    for type_match in data["typeMatches"]:
        for series in type_match.get("seriesMatches", []):
            if "seriesAdWrapper" in series:
                for match in series["seriesAdWrapper"].get("matches", []):
                    
                    match_info = match.get("matchInfo", {})
                    
                    team1 = match_info.get("team1", {}).get("teamName")
                    team2 = match_info.get("team2", {}).get("teamName")
                    venue = match_info.get("venueInfo", {}).get("ground")
                    
                    teams.update([team1, team2])
                    venues.add(venue)
                    
                    matches.append(match_info)
                    
                    # Count venue usage
                    venue_count[venue] = venue_count.get(venue, 0) + 1

# ---------------------------
# Metrics
# ---------------------------
total_matches = len(matches)
total_teams = len(teams)
total_venues = len(venues)

top_team = max(teams, key=lambda x: list(teams).count(x)) if teams else "N/A"
top_venue = max(venue_count, key=venue_count.get) if venue_count else "N/A"

# ---------------------------
# UI Cards
# ---------------------------
col1, col2, col3, col4, col5 = st.columns(5)

def card(title, value):
    st.markdown(
        f"""
        <div style="
            background-color:#1f2c3a;
            padding:20px;
            border-radius:10px;
            text-align:center;
            color:white;
        ">
            <h4>{title}</h4>
            <h2>{value}</h2>
        </div>
        """,
        unsafe_allow_html=True
    )

with col1:
    card("Matches", total_matches)

with col2:
    card("Teams", total_teams)

with col3:
    card("Venues", total_venues)

with col4:
    card("Top Team", top_team)

with col5:
    card("Top Venue", top_venue)

# ---------------------------
# Match Table
# ---------------------------
st.markdown("## Live Matches")

df = pd.DataFrame([
    {
        "Match": m.get("matchDesc"),
        "Team 1": m.get("team1", {}).get("teamName"),
        "Team 2": m.get("team2", {}).get("teamName"),
        "Venue": m.get("venueInfo", {}).get("ground")
    }
    for m in matches
])

st.dataframe(df)

## Live matches